# 01 - Generate synthetic images

PRD sections 3.4-3.6 and 3.11, flow 2. Order: attribute list from an LLM, you review it, then method A (add objects) and method B (AI scene edits).

**Before you start:** the real photos and their hand-drawn boxes must be in Drive: `real/*.jpg` and `real/annotations.json` (COCO, class names as in `Data/objects_table.csv`). Generation starts only from the *train* part of `real/split.json`, which this notebook creates once with the seed.

Output (format in the README): `synthetic/images/<id>.png`, `synthetic/metadata.jsonl` (images that passed the filter), `synthetic/rejected.jsonl` (images the filter dropped, with the reason).

In [ ]:
# Colab's preinstalled torchao is older than what diffusers/peft expect; we don't use it.
%pip uninstall -q -y torchao
%pip install -q diffusers==0.35.1 transformers peft accelerate openai

In [ ]:
BRANCH = "dev"  # set to your branch name to run a branch

import os, subprocess, yaml
REPO = "/content/genai-shelter-validator"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/AlmogSchwartzberg/genai-shelter-validator.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
cfg = yaml.safe_load(open(f"{REPO}/config.yaml"))

from google.colab import drive
drive.mount("/content/drive")
ROOT = cfg["drive_root"]
os.makedirs(ROOT, exist_ok=True)
print("Config loaded, Drive ready at", ROOT)

## 1. Load the objects table and background images

In [ ]:
import sys, json, random
import torch
from PIL import Image, ImageDraw
from diffusers.utils import make_image_grid
sys.path.insert(0, f"{REPO}/Code")
import synth

PILOT = True  # True: 2 images per class and edit type, to check the whole pipeline. False: the full run. Pilot images are reused by the full run.
if PILOT:
    cfg["generation"].update(images_per_class=2, images_per_edit_type=2, overshoot=1.0)
g = cfg["generation"]

rows = synth.load_objects(f"{REPO}/Data/objects_table.csv")
names = [r["name"] for r in rows]
real_dir = f"{ROOT}/real"
items = synth.load_real(real_dir, names, g["image_size"])
split = synth.load_split(real_dir, items, cfg["seed"])
print(f"{len(items)} labeled real photos; split:", {k: len(v) for k, v in split.items()})
print("Generation uses only the train part. split.json is fixed once created: add all real photos before the first run.")

## 1b. Attribute list from an LLM (PRD 3.11)

Run once. The list feeds every prompt. The working copy is `synthetic/attributes.json` in Drive, so it survives a runtime reset. **Read it, fix it, then set `"reviewed": true` yourself** (open it from the Drive folder). Generation stops until `reviewed` is true. When the team approves it, commit that file as `Data/attributes.json`; a committed copy is restored to Drive if Drive has none. Check the warnings: a scene-edit phrase that names a light, picture, mirror or bottle makes the model paint an object that has no box.

The key comes from the Colab secret (or env var) `OPENAI_API_KEY`. Other providers: set `llm.base_url` and `llm.model` in `config.yaml` (any OpenAI-compatible API).

In [ ]:
import shutil
ATTR = f"{ROOT}/synthetic/attributes.json"  # working copy, on Drive
os.makedirs(f"{ROOT}/synthetic", exist_ok=True)
REGENERATE = False  # True overwrites the file, including a reviewed one

if not REGENERATE and not os.path.exists(ATTR) and os.path.exists(f"{REPO}/Data/attributes.json"):
    shutil.copy(f"{REPO}/Data/attributes.json", ATTR)  # the approved copy committed to Git
    print("Restored the committed Data/attributes.json to Drive")
if REGENERATE or not os.path.exists(ATTR):
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            key = None
    if not key:
        raise RuntimeError("No LLM key: add the Colab secret OPENAI_API_KEY (key icon, left bar) or set the env var")
    attrs, warnings = synth.generate_attributes(cfg, rows, key)
    json.dump(attrs, open(ATTR, "w"), indent=2)
    print("Saved", ATTR, "- review it, then set reviewed to true by hand")
    print("Phrases with a negation (image models ignore these, rewrite them):", warnings or "none")
else:
    print(ATTR, "already exists, not regenerated")
print(open(ATTR).read())

In [ ]:
attrs = synth.load_attributes(ATTR, names)  # stops here until "reviewed": true
print("Attributes reviewed:", {k: len(v) for k, v in attrs.items() if isinstance(v, list)})

## 2. Method A: add objects with Stable Diffusion inpainting (boxes saved automatically)

A free spot is picked in a real *train* photo (never on top of an existing box), sized from the objects table, and the box itself is the inpainting mask. The box is the label. Lookalikes use the same call, with the missing part (hose, cover) in the negative prompt. `placement` in `config.yaml` selects plain inpainting, ControlNet or GLIGEN.

Green = existing real boxes (kept as labels), red = where the object will be painted.

In [ ]:
jobs = synth.plan_jobs(cfg, rows, attrs, items, split)
a_jobs = [j for j in jobs if j["method"] == "add_objects"]
b_jobs = [j for j in jobs if j["method"] == "scene_edits"]
print(len(a_jobs), "add_objects jobs,", len(b_jobs), "scene_edits jobs planned (a rerun skips what is already saved)")

def preview(sel):
    tiles = []
    for j in sel:
        im = Image.open(f"{real_dir}/{j['source']}").convert("RGB").resize((256, 256))
        d = ImageDraw.Draw(im)
        for b in j["source_boxes"]: d.rectangle([v / 2 for v in b["xyxy"]], outline="lime", width=2)
        for b in j["paint_boxes"]: d.rectangle([v / 2 for v in b], outline="red", width=2)
        tiles.append(im)
        print(j["id"], "|", j["prompt"])
    return make_image_grid(tiles, 1, len(tiles))

preview(random.Random(0).sample(a_jobs, 4))

## 3. Method B: AI scene edits - add people, furniture, change room size and style (boxes protected)

Only train photos are edited, and nothing is painted in the gray border of a fitted photo. The painted area never includes a box: `add_people` and `add_furniture` paint free regions in the lower half, `room_style` repaints everything outside the boxes and an 8 px margin around them (strength 0.75), `room_size` shrinks the photo into a bigger canvas and outpaints the border, and the boxes are rescaled by the known factor. After generation, any image where the model changed pixels inside a box by more than `box_change_threshold` is dropped, and the box pixels are copied back from the original. The class objects are in the negative prompt of every scene edit, because a new light, picture or bottle would have no box.

**In the pilot, look at the scene-edit samples** (section 5) for exactly that: new lights, pictures, mirrors, bottles, extinguishers or cylinders with no box. Note how many of the 8 you see, and report it.

In [ ]:
preview(random.Random(0).sample([j for j in b_jobs if j["edit_type"] in synth.REGIONS], 4))

## 4. Generate, save each image and its record to Drive (resumes after a disconnect)

Every image gets a fixed seed, so a rerun gives the same image. After a disconnect, run the notebook again from the top: images that already have a record are skipped.

In [ ]:
paint = synth.make_painter(cfg)
from transformers import CLIPTokenizer
tok = CLIPTokenizer.from_pretrained(g["model"], subfolder="tokenizer")
too_long = synth.long_prompts(jobs, tok)
print("Prompts longer than 77 tokens (the end is cut):", too_long or "none")

import time
start = time.time()
made = synth.run(cfg, ROOT, jobs, paint)
print(f"{made} new images in {(time.time() - start) / 60:.1f} min")

## 5. Show 16 random samples and counts per object type

In [ ]:
import pandas as pd
cov = synth.coverage(ROOT, rows)
target = g["images_per_class"]
table = pd.DataFrame({"passed images": cov["per_class"]})
table["status"] = ["ok" if c >= target else f"BELOW {target}" for c in table["passed images"]]
display(table)
print("scene edits that passed:", cov["per_edit_type"], "(target", g["images_per_edit_type"], "each)")
print("why images were dropped:", cov["rejected_reasons"] or "none")
print("A class or edit type below target: raise overshoot in config.yaml and run again, only the new images are made.")

In [ ]:
recs = [json.loads(l) for l in open(f"{ROOT}/synthetic/metadata.jsonl")]
tiles = []
for r in random.Random(0).sample(recs, min(16, len(recs))):
    im = Image.open(f"{ROOT}/{r['image']}").convert("RGB")
    d = ImageDraw.Draw(im)
    for b in r["boxes"]:
        d.rectangle(b["xyxy"], outline="red" if b["origin"] == "synthetic" else "lime", width=3)
        d.text((b["xyxy"][0] + 3, b["xyxy"][1] + 3), b["class"], fill="yellow")
    tiles.append(im.resize((256, 256)))
make_image_grid(tiles, 4, 4)